# PVP 三聚体 · 同时处理 C5 与主链两个起点
在 Google Colab 选择 **CPU 运行时 → 运行所有单元格**。输入结构和工作流已内置。程序依次运行 C5、BB，输出各自的实际反应网络、候选结构和日志。浏览器出现下载提示时保存 ZIP。运行时间取决于分配的 CPU、编译和采样。若一个位点未发现可用通道，程序仍尝试另一个位点。

默认先做小批量采样，两边都结束后，只给缺少候选且成功启动搜索的位点补跑一次。结果记录在 `colab_status.json`。计算仅覆盖 AutoMeKin-X 低层搜索。DFT 与双向 IRC 可在有 ORCA 6 的环境上从结果包继续。

In [ ]:
# 解包内置的源程序与两份真实 XYZ，无需再上传 ZIP。
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime
import base64, hashlib, io, json, os, shutil, subprocess, sys, tempfile, time
from google.colab import files

bundle = base64.b64decode("...")
assert hashlib.sha256(bundle).hexdigest() == "28120867792c06464df284bd773782cbb0ae844fda6535c02fe38a1ebde6578e"
workspace = Path(tempfile.mkdtemp(dir="/content", prefix="pvp_two_sites_"))
with ZipFile(io.BytesIO(bundle)) as archive:
    archive.extractall(workspace)
project = workspace / "PVP3_workflow"
assert (project / "run.py").is_file()
print("计算目录", project)
subprocess.run([sys.executable, "run.py", "check"], cwd=project, check=True)

In [ ]:
# 记录完整安装日志，并保留从 GitHub 安装的源程序版本。
def run_logged(command, logfile):
    # 计算时使用后台子进程，等待真实退出码。安装日志保存在磁盘。
    print("启动", " ".join(map(str,command)), flush=True)
    with logfile.open("w") as out:
        proc = subprocess.Popen(command, cwd=project, stdout=out, stderr=subprocess.STDOUT)
        since = time.monotonic()
        while proc.poll() is None:
            try:
                proc.wait(timeout=30)
            except subprocess.TimeoutExpired:
                print("已运行 %.1f 分钟；日志 %s" % ((time.monotonic()-since)/60, logfile.name), flush=True)
    if proc.returncode:
        with logfile.open("rb") as src:
            src.seek(max(0,logfile.stat().st_size-2200))
            print(src.read().decode(errors="replace"))
    return proc.returncode

if not (shutil.which("llcalcs.sh") and shutil.which("select.sh")):
    install_rc = run_logged(["bash", str(project/"colab/install.sh")], project/"install.log")
    if install_rc:
        files.download(str(project/"install.log"))
        raise RuntimeError("安装没有完成。已保存 install.log，后面的搜索单元不会启动。")
    prefix = "/content/pvp_amk_install"
    os.environ["AMK"] = prefix
    os.environ["PATH"] = ":".join(prefix+p for p in ["/bin","/bin/HLscripts","/bin/MOPAC_DEV"]) + ":" + os.environ["PATH"]
    for var in ("LD_LIBRARY_PATH","LIBRARY_PATH"):
        os.environ[var] = prefix+"/lib:"+prefix+"/bin/MOPAC_DEV:"+os.environ.get(var,"")
os.environ["inter"]="0"
assert shutil.which("llcalcs.sh") and shutil.which("select.sh"), "缺少 AutoMeKin-X 命令"
print("AutoMeKin-X 已就绪。")

In [ ]:
# 可在每个任务结束后查看左侧“文件”。两位点都结束后下载一个汇总包。
assert shutil.which("llcalcs.sh") and shutil.which("select.sh"), "请先完成安装单元"
import re

automatic_retry = True  # @param {type:"boolean"}
settings=project/"settings.json"
original=settings.read_text()
dat_paths={s:project/"seeds"/(s+".dat") for s in ("C5","BB")}
original_dat={s:p.read_text() for s,p in dat_paths.items()}
status={s:[] for s in dat_paths}

def record():
    (project/"colab_status.json").write_text(json.dumps(status,indent=2,ensure_ascii=False))

def search_site(site, phase):
    path=project/"runs"/site/"search"
    before=set(path.glob("batch_*"))
    stamp=datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    log=project/f"{site}_{phase}_{stamp}.log"
    rc=run_logged([sys.executable,"-u","run.py","search",site],log)
    new=sorted(set(path.glob("batch_*"))-before)
    info=dict(phase=phase,returncode=rc,log=log.name,candidates=0,
              batch=str(new[-1].relative_to(project)) if new else None)
    if rc==0:
        root=project/"runs"/site
        key=json.loads((root/"current.json").read_text())["snapshot"]
        snap=root/"catalogs"/key
        entries=json.loads((snap/"edges.json").read_text())
        info.update(snapshot=key,raw_channels=len(entries),candidates=sum(bool(e["usable"]) for e in entries))
    if new:
        rxnet=new[-1]/f"FINAL_LL_{site}"/"RXNet"
        info["rxnet_exists"]=rxnet.is_file()
        rawlog=new[-1]/"ll.log"
        if rawlog.is_file() and not info["candidates"]:
            with rawlog.open("rb") as stream:
                stream.seek(max(0,rawlog.stat().st_size-1200))
                print(stream.read().decode(errors="replace"))
    status[site].append(info)
    record()
    print(site,phase,"可用低层候选",info["candidates"],"；批次",info["batch"],flush=True)

def save_zip():
    have=any(attempt["candidates"] for trials in status.values() for attempt in trials)
    label="LL_candidates" if have else "search_diagnostics"
    out=workspace/("PVP3_"+label+"_"+datetime.now().strftime("%Y%m%d_%H%M%S")+".zip")
    with ZipFile(out,"w",compression=ZIP_DEFLATED,compresslevel=3) as z:
        for p in sorted(project.rglob("*")):
            if not p.is_file() or "__pycache__" in p.parts or p.name==".run.lock":
                continue
            # The curated finals, native DBs and logs enable re-import, while temporary MD batches can be large.
            rel=p.relative_to(project)
            if "search" in rel.parts and not any(k.startswith("FINAL_LL_") for k in rel.parts):
                if p.name not in {"ll.log","command.json","C5.dat","C5.xyz","BB.dat","BB.xyz"}:
                    continue
            z.write(p,p.relative_to(workspace))
    print("下载文件",out,flush=True)
    files.download(str(out))

try:
    for site in dat_paths:
        search_site(site,"pilot")
    if automatic_retry:
        for site in dat_paths:
            trial=status[site][-1]
            if trial["candidates"] or not trial.get("rxnet_exists"):
                continue
            cfg=json.loads(original)
            cfg.update(tasks=4,iterations=3,parallel=min(2,os.cpu_count() or 1))
            settings.write_text(json.dumps(cfg,indent=2))
            dat_paths[site].write_text(re.sub(r"(?m)^ntraj\s+\d+\s*$","ntraj 5",original_dat[site]))
            search_site(site,"expanded")
finally:
    settings.write_text(original)
    for site,p in dat_paths.items():
        p.write_text(original_dat[site])
    record()
    save_zip()

下载名称含 `LL_candidates` 表示至少一个位点有可导出的低层候选。分别打开 `colab_status.json` 查看 C5、BB 的数量，再查看对应 `candidates.csv` 和 XYZ。名称含 `search_diagnostics` 表示未导出可用候选，可把它交回分析失败原因。未开始搜索时如安装失败，请保留 `install.log`。